In [1]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import warnings

warnings.filterwarnings('ignore')

print("\n--- 1. VERİ YÜKLEME VE SİHİRLİ ÖZELLİKLER (MAGIC FEATURES) ---")

train = pd.read_csv("../data/raw/train.csv", index_col='id')
test = pd.read_csv("../data/raw/test.csv", index_col='id')

y = train['Churn'].map({'Yes': 1, 'No': 0})
train.drop('Churn', axis=1, inplace=True)

# Eğitim ve test setini aynı anda işlemek için birleştiriyoruz
df = pd.concat([train, test])
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)

# -------------------------------------------------------------------
# KAGGLE SİHİRLİ ÖZELLİKLERİ (MAGIC FEATURES)
# -------------------------------------------------------------------

# 1. Sentetik Veri Sızıntısı (Artifact): Normalde TotalCharges = MonthlyCharges * tenure olmalıdır. 
# Sentetik verilerde bu fark (hata payı) model için inanılmaz bir ipucu (leak) olabilir!
df['TotalCharges_Diff'] = df['TotalCharges'] - (df['MonthlyCharges'] * df['tenure'])

# 2. Hizmet Bağımlılığı (Lock-in): Müşteri ne kadar çok ek hizmet alırsa (Güvenlik, Yedekleme, TV vs.) o kadar zor ayrılır.
ek_hizmetler = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
df['Total_Extra_Services'] = (df[ek_hizmetler] == 'Yes').sum(axis=1)

# 3. Sözleşme / Süre Oranı: Aydan aya ödeyen ile 2 yıllık ödeyenin 'tenure' değeri farklı ağırlıktadır.
contract_map = {'Month-to-month': 1, 'One year': 12, 'Two year': 24}
df['Contract_Months'] = df['Contract'].map(contract_map)
# Müşteri sözleşmesinin kaçıncı katında? (Örn: 2 yıllık sözleşmesi olup 24 aydır buradaysa oran 1'dir)
df['Tenure_Contract_Ratio'] = df['tenure'] / df['Contract_Months']

# 4. Otomatik Ödeme Kolaylığı: Kredi kartı veya banka havalesiyle "otomatik" ödeyenler iptal etmeye üşenir.
df['Auto_Payment'] = df['PaymentMethod'].isin(['Bank transfer (automatic)', 'Credit card (automatic)']).astype(int)

# 5. Aile/Bağ Sorumluluğu: Partneri veya bakmakla yükümlü olduğu biri olanlar daha stabil müşterilerdir.
df['Has_Family'] = ((df['Partner'] == 'Yes') | (df['Dependents'] == 'Yes')).astype(int)

# -------------------------------------------------------------------

# Kategorik değişkenleri One-Hot yapıyoruz
cat_cols = df.select_dtypes(include=['object']).columns
df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True)

# Veriyi tekrar ayırıyoruz
X_train = df_encoded.iloc[:len(train)]
X_test = df_encoded.iloc[len(train):]

print(f"Toplam özellik sayımız {X_train.shape[1]}'e çıktı! Eğitim başlıyor...\n")

# --- 2. MODEL EĞİTİMİ (XGBOOST) ---
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(X_train))
test_preds = np.zeros(len(X_test))

for fold, (train_idx, valid_idx) in enumerate(skf.split(X_train, y)):
    X_tr, y_tr = X_train.iloc[train_idx], y.iloc[train_idx]
    X_va, y_va = X_train.iloc[valid_idx], y.iloc[valid_idx]
    
    model = XGBClassifier(
        n_estimators=600,
        learning_rate=0.03,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric="auc"
    )
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    
    valid_preds = model.predict_proba(X_va)[:, 1]
    oof_preds[valid_idx] = valid_preds
    test_preds += model.predict_proba(X_test)[:, 1] / skf.n_splits
    
    print(f"Magic Fold {fold+1} AUC: {roc_auc_score(y_va, valid_preds):.4f}")

cv_auc = roc_auc_score(y, oof_preds)
print(f"\n🏆 Sihirli Özellikler (OOF) AUC Skoru: {cv_auc:.4f}")

# Gönderim Dosyası
submission = pd.DataFrame({'id': test.index, 'Churn': test_preds})
submission.to_csv("../submissions/submission_magic_features.csv", index=False)
print("✅ Yeni veri mühendisliği dosyası 'submissions' klasörüne kaydedildi!")


--- 1. VERİ YÜKLEME VE SİHİRLİ ÖZELLİKLER (MAGIC FEATURES) ---
Toplam özellik sayımız 36'e çıktı! Eğitim başlıyor...

Magic Fold 1 AUC: 0.9159
Magic Fold 2 AUC: 0.9169
Magic Fold 3 AUC: 0.9164
Magic Fold 4 AUC: 0.9175
Magic Fold 5 AUC: 0.9147

🏆 Sihirli Özellikler (OOF) AUC Skoru: 0.9163
✅ Yeni veri mühendisliği dosyası 'submissions' klasörüne kaydedildi!


In [2]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import warnings

warnings.filterwarnings('ignore')

print("\n--- 2. MEGA VERİ HAMLESİ: SİHİRLİ ÖZELLİKLER + ORİJİNAL VERİ ---")

# 1. Verileri Yüklüyoruz
train = pd.read_csv("../data/raw/train.csv", index_col='id')
test = pd.read_csv("../data/raw/test.csv", index_col='id')

# Orijinal Veriyi Çekiyoruz (Kaggle Forumundan)
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
orig = pd.read_csv(url)
orig = orig.rename(columns={'customerID': 'id'}).set_index('id')

# Hedef Değişkenleri Ayırıyoruz
y_train = train['Churn'].map({'Yes': 1, 'No': 0})
train.drop('Churn', axis=1, inplace=True)

y_orig = orig['Churn'].map({'Yes': 1, 'No': 0})
orig.drop('Churn', axis=1, inplace=True)

# Eğitim için Train ve Orig'i alt alta birleştiriyoruz
X_train_full = pd.concat([train, orig])
y_train_full = pd.concat([y_train, y_orig])

# Tüm veriyi (Test dahil) Feature Engineering için birleştiriyoruz
df_all = pd.concat([X_train_full, test], keys=['train', 'test'])
df_all['TotalCharges'] = pd.to_numeric(df_all['TotalCharges'], errors='coerce').fillna(0)

# -------------------------------------------------------------------
# 2. SİHİRLİ ÖZELLİKLERİ UYGULUYORUZ
# -------------------------------------------------------------------
df_all['TotalCharges_Diff'] = df_all['TotalCharges'] - (df_all['MonthlyCharges'] * df_all['tenure'])

ek_hizmetler = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
df_all['Total_Extra_Services'] = (df_all[ek_hizmetler] == 'Yes').sum(axis=1)

contract_map = {'Month-to-month': 1, 'One year': 12, 'Two year': 24}
df_all['Contract_Months'] = df_all['Contract'].map(contract_map)
df_all['Tenure_Contract_Ratio'] = df_all['tenure'] / df_all['Contract_Months']

df_all['Auto_Payment'] = df_all['PaymentMethod'].isin(['Bank transfer (automatic)', 'Credit card (automatic)']).astype(int)
df_all['Has_Family'] = ((df_all['Partner'] == 'Yes') | (df_all['Dependents'] == 'Yes')).astype(int)

# One-Hot Encoding
cat_cols = df_all.select_dtypes(include=['object']).columns
df_all_encoded = pd.get_dummies(df_all, columns=cat_cols, drop_first=True)

# İşlem bitince veriyi tekrar Train ve Test olarak ayırıyoruz
X_train_final = df_all_encoded.xs('train')
X_test_final = df_all_encoded.xs('test')

print(f"Gerçek Dünya Verisi Eklendi! Yepyeni Eğitim Seti Boyutu: {X_train_final.shape}")
print("Eğitim başlıyor...\n")

# -------------------------------------------------------------------
# 3. XGBOOST EĞİTİMİ (5-Fold)
# -------------------------------------------------------------------
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(X_train_final))
test_preds = np.zeros(len(X_test_final))

for fold, (train_idx, valid_idx) in enumerate(skf.split(X_train_final, y_train_full)):
    X_tr, y_tr = X_train_final.iloc[train_idx], y_train_full.iloc[train_idx]
    X_va, y_va = X_train_final.iloc[valid_idx], y_train_full.iloc[valid_idx]
    
    model = XGBClassifier(
        n_estimators=600,
        learning_rate=0.03,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric="auc"
    )
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    
    valid_preds = model.predict_proba(X_va)[:, 1]
    oof_preds[valid_idx] = valid_preds
    test_preds += model.predict_proba(X_test_final)[:, 1] / skf.n_splits
    
    print(f"Magic+Orig Fold {fold+1} AUC: {roc_auc_score(y_va, valid_preds):.4f}")

cv_auc = roc_auc_score(y_train_full, oof_preds)
print(f"\n🏆 Sihirli Özellikler + Orijinal Veri OOF Skoru: {cv_auc:.4f}")

# Gönderim Dosyası
submission = pd.DataFrame({'id': X_test_final.index, 'Churn': test_preds})
submission.to_csv("../submissions/submission_magic_plus_orig.csv", index=False)
print("✅ Mega Veri gönderim dosyası 'submissions' klasörüne kaydedildi!")


--- 2. MEGA VERİ HAMLESİ: SİHİRLİ ÖZELLİKLER + ORİJİNAL VERİ ---
Gerçek Dünya Verisi Eklendi! Yepyeni Eğitim Seti Boyutu: (601237, 36)
Eğitim başlıyor...

Magic+Orig Fold 1 AUC: 0.9159
Magic+Orig Fold 2 AUC: 0.9164
Magic+Orig Fold 3 AUC: 0.9140
Magic+Orig Fold 4 AUC: 0.9152
Magic+Orig Fold 5 AUC: 0.9162

🏆 Sihirli Özellikler + Orijinal Veri OOF Skoru: 0.9155
✅ Mega Veri gönderim dosyası 'submissions' klasörüne kaydedildi!


In [3]:
from catboost import CatBoostClassifier

print("\n--- 3. CATBOOST İLE SİHİRLİ ÖZELLİKLER EĞİTİMİ (5-Fold) ---")

# Tahminleri tutacağımız diziler
oof_preds_cb_magic = np.zeros(len(X_train_final))
test_preds_cb_magic = np.zeros(len(X_test_final))

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, valid_idx) in enumerate(skf.split(X_train_final, y_train_full)):
    X_tr, y_tr = X_train_final.iloc[train_idx], y_train_full.iloc[train_idx]
    X_va, y_va = X_train_final.iloc[valid_idx], y_train_full.iloc[valid_idx]
    
    # CatBoost'u sihirli verimize uygun ayarlarla kuruyoruz
    model_cb_magic = CatBoostClassifier(
        iterations=700,
        learning_rate=0.03,
        depth=6,
        eval_metric='AUC',
        random_seed=42,
        verbose=False
    )
    
    model_cb_magic.fit(
        X_tr, y_tr, 
        eval_set=(X_va, y_va), 
        early_stopping_rounds=50, 
        verbose=False
    )
    
    valid_preds = model_cb_magic.predict_proba(X_va)[:, 1]
    oof_preds_cb_magic[valid_idx] = valid_preds
    
    test_preds_cb_magic += model_cb_magic.predict_proba(X_test_final)[:, 1] / skf.n_splits
    
    print(f"CatBoost Magic Fold {fold+1} AUC: {roc_auc_score(y_va, valid_preds):.4f}")

cv_auc_cb = roc_auc_score(y_train_full, oof_preds_cb_magic)
print(f"\n🚀 CatBoost (Magic+Orig) OOF Skoru: {cv_auc_cb:.4f}")

# Gönderim Dosyası
submission_cb_magic = pd.DataFrame({'id': X_test_final.index, 'Churn': test_preds_cb_magic})
submission_cb_magic.to_csv("../submissions/submission_catboost_magic.csv", index=False)
print("✅ CatBoost Magic gönderim dosyası 'submissions' klasörüne kaydedildi!")


--- 3. CATBOOST İLE SİHİRLİ ÖZELLİKLER EĞİTİMİ (5-Fold) ---
CatBoost Magic Fold 1 AUC: 0.9151
CatBoost Magic Fold 2 AUC: 0.9154
CatBoost Magic Fold 3 AUC: 0.9133
CatBoost Magic Fold 4 AUC: 0.9146
CatBoost Magic Fold 5 AUC: 0.9154

🚀 CatBoost (Magic+Orig) OOF Skoru: 0.9148
✅ CatBoost Magic gönderim dosyası 'submissions' klasörüne kaydedildi!
